# 🚀 Mockmate-LLM — Free QLoRA Fine-tuning on Google Colab (T4 16GB)

This notebook fine-tunes `deepseek-ai/deepseek-coder-6.7b-base` on the
**greengerong/leetcode** dataset (2,359 problems × 4 languages = 9,438 examples)
using **QLoRA (4-bit NF4)** to fit in Colab's free T4 GPU.

## What you'll get
A LoRA adapter (~80 MB) that you can merge into the base model and serve via
the FastAPI backend in `mockmate-llm/app_api.py`.

## Cost & time
- **Cost**: $0 (Colab Free tier)
- **Time**: ~6-9 hours for 3 epochs (Colab may disconnect after 12h — be sure to save checkpoints to Google Drive)
- **GPU**: Tesla T4 16GB (Colab Free) or A100 40GB (Colab Pro)

## Memory budget (T4 16GB)
| Component | Footprint |
|---|---|
| Base weights (4-bit NF4) | ~3.5 GB |
| LoRA params (r=8) | ~40 MB |
| Activations (bs=1, seq=512) | ~6 GB |
| Optimizer (8-bit paged AdamW) | ~1 GB |
| Buffers | ~2 GB |
| **Total** | **~12.5 GB** (fits in 16 GB) |

**If you OOM**: lower `MAX_SEQ_LEN` to 384 or `BATCH_SIZE` to 1 with `GRAD_ACCUM=16`.

## Step 1 — Enable GPU & check

**Runtime → Change runtime type → Hardware accelerator → T4 GPU**
(Should already be applied via the notebook metadata.)

In [ ]:
import torch
assert torch.cuda.is_available(), '❌ No GPU. Runtime → Change runtime type → T4 GPU.'
print(f'✅ GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')
!nvidia-smi -L

## Step 2 — Install dependencies (≈ 5 min)

Pinned versions known to work on Colab's CUDA 12.x runtime.

In [ ]:
# Pin versions that are known to co-exist on Colab (CUDA 12.1)
!pip install -q torch==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q \
    transformers==4.44.2 \
    tokenizers==0.19.1 \
    datasets==2.21.0 \
    accelerate==0.33.0 \
    peft==0.12.0 \
    trl==0.9.6 \
    bitsandbytes==0.43.3 \
    sentencepiece==0.2.0 \
    safetensors==0.4.4 \
    huggingface_hub==0.24.6 \
    einops==0.8.0 \
    tensorboard==2.17.0
# Optional speedup — install flash-attn (only works on Ampere+, ~3 min build)
# !pip install -q flash-attn==2.6.3 --no-build-isolation
print('✅ Install done.')

## Step 3 — Login to Hugging Face (needed to download DeepSeek-Coder)

1. Create a free HF account at https://huggingface.co/join
2. Go to https://huggingface.co/settings/tokens → New token → Read access → copy
3. Paste when prompted below.

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted

## Step 4 — Mount Google Drive (for checkpoint saves)

Colab can disconnect after ~12 hours of inactivity. Saving checkpoints to
Google Drive means you can resume training without losing progress.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK_DIR = '/content/drive/MyDrive/mockmate-llm'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)
print(f'✅ Drive mounted. Checkpoints will save to: {WORK_DIR}/checkpoints/')

## Step 5 — Clone the project & prepare the dataset

Replace `Rishabh01487` with your GitHub username. If you haven't pushed
the mockmate-llm repo yet, fork from your local copy to a new GitHub repo
and update the URL below.

In [ ]:
import os
os.chdir('/content')
# TODO: replace with your own repo URL after forking
!git clone https://github.com/Rishabh01487/mockmate-llm.git 2>/dev/null || echo 'Repo already cloned'
os.chdir('/content/mockmate-llm')
!git pull 2>/dev/null || true
print('✅ Repo cloned.')
print('Files:', os.listdir('.'))

In [ ]:
# Download the greengerong/leetcode dataset directly from Hugging Face
import urllib.request, os
os.chdir('/content/mockmate-llm')
url = 'https://huggingface.co/datasets/greengerong/leetcode/resolve/main/leetcode-train.jsonl'
out = '/tmp/greengerong.jsonl'
if not os.path.exists(out):
    print(f'Downloading {url} ...')
    urllib.request.urlretrieve(url, out)
print(f'✅ Dataset downloaded: {os.path.getsize(out) / 1e6:.1f} MB')

# Convert to our JSONL schema (one row per problem-language pair)
!python convert_greengerong_dataset.py --input /tmp/greengerong.jsonl --out ./leetcode_dataset.jsonl
print()
print('=== Running prepare_data.py ===')
!python prepare_data.py \
    --input ./leetcode_dataset.jsonl \
    --out_dir ./data_processed \
    --val_size 0.05 \
    --seed 42

## Step 6 — Verify dataset

Should show ~8,900 train + ~460 val examples, balanced across 4 languages.

In [ ]:
import json
stats = json.load(open('./data_processed/stats.json'))
print(f"Train: {stats['train_count']}")
print(f"Val:   {stats['val_count']}")
print(f"Languages:    {stats['language_distribution']}")
print(f"Difficulties: {stats['difficulty_distribution']}")
print(f"Top complexities: {dict(sorted(stats['time_complexity_distribution'].items(), key=lambda x:-x[1])[:5])}")
assert stats['train_count'] > 5000, 'Dataset too small!'
print('✅ Dataset looks good.')

## Step 7 — Train! (≈ 6-9 hours for 3 epochs)

### Config tuned for T4 16GB free tier
| Param | Value | Why |
|---|---|---|
| `--batch_size 1` | 1 | T4 VRAM is tight |
| `--grad_accum 16` | 16 | Effective batch = 16 (same as 4×4 on A100) |
| `--max_seq_len 512` | 512 | 90% of examples fit; longest are trimmed |
| `--lora_r 8` | 8 | Half of full config to save memory |
| `--lora_alpha 16` | 16 | Standard 2× ratio |
| `--epochs 3` | 3 | Standard for SFT on 8-10K examples |
| `--save_steps 100` | 100 | Checkpoint every ~45 min |

**If you disconnect**: just re-run Steps 1-6, then run the cell below with `RESUME=1`.

In [ ]:
import os
os.chdir('/content/mockmate-llm')

# Set environment for the training run
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = '/content/hf_cache'  # cache model downloads

RESUME = 0  # Set to 1 to resume from latest checkpoint
resume_arg = f'--resume_from_checkpoint {WORK_DIR}/checkpoints/deepseek-leetcode-qlora' if RESUME else ''

!python train_qlora.py \
    --data_dir ./data_processed \
    --base_model deepseek-ai/deepseek-coder-6.7b-base \
    --output_dir "{WORK_DIR}/checkpoints/deepseek-leetcode-qlora" \
    --epochs 3 \
    --batch_size 1 \
    --grad_accum 16 \
    --lr 2e-4 \
    --max_seq_len 512 \
    --lora_r 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --warmup_ratio 0.03 \
    --save_steps 100 \
    --eval_steps 100 \
    --logging_steps 10 \
    --bf16 \
    --seed 42 \
    {resume_arg}

## Step 8 — Smoke test the adapter (1 minute)

Verifies the trained LoRA actually generates sensible code.

In [ ]:
os.chdir('/content/mockmate-llm')
!python inference.py \
    --adapter "{WORK_DIR}/checkpoints/deepseek-leetcode-qlora" \
    --title "Two Sum" \
    --difficulty Easy \
    --language python \
    --description "Given an array of integers nums and an integer target, return indices of the two numbers such that they add up to target." \
    --examples "Input: nums = [2,7,11,15], target = 9 -> Output: [0,1]" \
    --constraints "2 <= nums.length <= 10^4"

## Step 9 — Push to HuggingFace Hub (optional but recommended)

Uploading to HF Hub means you can load it from any machine later without
carrying the weights around.

In [ ]:
# Replace YOUR_HF_USERNAME with your HF username
HF_REPO = 'YOUR_HF_USERNAME/deepseek-leetcode-qlora'  # ← change this

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id=HF_REPO, repo_type='model', exist_ok=True)
api.upload_folder(
    folder_path=f'{WORK_DIR}/checkpoints/deepseek-leetcode-qlora',
    repo_id=HF_REPO,
    repo_type='model',
)
print(f'✅ Uploaded to https://huggingface.co/{HF_REPO}')

## Step 10 — Download the adapter to your local machine

If you didn't push to HF Hub, you can download the adapter directly from
Google Drive to your local machine.

In [ ]:
import os, shutil
# Zip the adapter for easy download
adapter_dir = f'{WORK_DIR}/checkpoints/deepseek-leetcode-qlora'
zip_path = f'{WORK_DIR}/deepseek-leetcode-qlora.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive(zip_path[:-4], 'zip', adapter_dir)
print(f'✅ Zipped adapter: {zip_path}')
print(f'   Size: {os.path.getsize(zip_path) / 1e6:.1f} MB')
print()
print('To download:')
print('  1. Open https://drive.google.com in your browser')
print('  2. Navigate to My Drive → mockmate-llm')
print('  3. Right-click deepseek-leetcode-qlora.zip → Download')

## ✅ Done — what's next?

You now have a fine-tuned LoRA adapter. To put it into production:

1. **Run the FastAPI backend** (on a GPU box):
   ```bash
   export ADAPTER_PATH=./checkpoints/deepseek-leetcode-qlora
   export MOCK_MODE=0 MODEL_PRELOAD=1
   uvicorn app_api:app --host 0.0.0.0 --port 8000
   ```

2. **Wire up Mockmate** — set `LLM_API_URL=http://<your-gpu-host>:8000` in
   `mockmate/backend/.env`.

3. **Test end-to-end** — open Mockmate, start a coding interview, click
   "⚡ Generate Solution" in the AI Copilot sidebar.

## Troubleshooting

| Issue | Fix |
|---|---|
| `CUDA out of memory` | Lower `--max_seq_len` to 384, or `--lora_r` to 4 |
| `KeyError: 'chatml_text'` | Re-run Step 5 — `prepare_data.py` failed silently |
| HuggingFace `401 Unauthorized` | Re-run Step 3 with a fresh token |
| `flash-attn` install fails | Skip it — SDPA is the default and works fine on T4 |
| Colab disconnects mid-training | Resume with `RESUME=1` in Step 7 (checkpoints save to Drive every 100 steps) |
| Training is too slow (>12h) | Reduce `--epochs` to 2 or filter `--max_rows 5000` in `prepare_data.py` |